# A Data-Centric Pipeline for Label Error Detection and Correction

**Question:** Can systematic detection and correction of mislabeled training examples improve
sentiment classification while the model architecture and training configuration stay fixed?

**Design:** IMDb train → inject known label noise of two kinds → Cleanlab (on frozen-BERT embeddings +
out-of-fold LogisticRegression) → clean (remove / hybrid) → fine-tune the *same* `bert-base-uncased` →
evaluate on the *same, untouched* test set.

**Two noise types:**
- **symmetric**: labels flipped uniformly at random (the textbook setting)
- **instance**: ambiguous reviews flipped far more often, like real annotator mistakes

## 1. Setup

In [ ]:
import os

REPO_URL = "https://github.com/MUKAMAFrancois/data-centric-ai.git"
REPO_DIR = "data-centric-ai"

if not os.path.exists("src"):
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    %cd {REPO_DIR}
!git pull -q
!pip install -q -r requirements.txt

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to GPU!")

### Persist results across disconnects
Free Colab sessions die. With `USE_DRIVE = True`, results and the embedding cache are written to Google Drive,
so re-running the notebook resumes where it stopped.

In [ ]:
from src.utils import load_config, set_seed

cfg = load_config("config.yaml")
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    base = "/content/drive/MyDrive/data-centric-ai"
    cfg["paths"].update({
        "cache_dir": f"{base}/cache",
        "results_file": f"{base}/experiments.csv",
        "issues_dir": f"{base}/label_issues",
    })

set_seed(cfg["seed"])
cfg["paths"]

In [ ]:
# Pilot switch: True for a quick check, False for the full experiment.
# Pilot runs are saved to the same results file, so the full run skips them.
PILOT = True

if PILOT:
    cfg["noise"]["types"] = ["instance"]
    cfg["noise"]["rates"] = [0.2]
    cfg["noise"]["seeds"] = [42]
    cfg["cleaning"]["strategies"] = ["remove"]

print("noise:", cfg["noise"], "| strategies:", cfg["cleaning"]["strategies"])

## 2. Load data

In [ ]:
from src.data import load_imdb, profile_data, split_overlap

train_df, test_df = load_imdb(
    cfg["data"]["dataset_name"],
    cfg["data"]["train_subset"],
    cfg["data"]["test_subset"],
    cfg["seed"])
print(f"train: {len(train_df):,}   test: {len(test_df):,}")
train_df.head()

## 3. Data profiling (baseline data quality)

In [ ]:
import pandas as pd

pd.DataFrame({"train": profile_data(train_df), "test": profile_data(test_df)})

In [ ]:
print("Exact train/test text overlap (leakage check):", split_overlap(train_df, test_df))

## 4. Controlled label noise
We flip the same fraction of labels in each class and keep the original label in `true_label`.
This known ground truth is what lets us grade Cleanlab objectively.

- **Symmetric** noise picks the reviews to flip uniformly at random.
- **Instance-dependent** noise picks them with probability proportional to how *ambiguous* each review is.
  Ambiguity = 1 − P(true label) from an out-of-fold TF-IDF + LogisticRegression "annotator" model, which is
  deliberately different from the BERT-based detector so the noise generator and detector stay independent.

In [ ]:
from src.data import ambiguity_scores, inject_noise

difficulty = ambiguity_scores(train_df["text"], train_df["label"], seed=cfg["seed"])

sym = inject_noise(train_df, 0.2, seed=42)
inst = inject_noise(train_df, 0.2, seed=42, noise_type="instance", difficulty=difficulty)
print(f"flipped: {inst.is_noisy.sum():,} of {len(inst):,}  |  class balance after: {inst.label.mean():.3f}")
print(f"mean ambiguity of flipped reviews  ->  random: {difficulty[sym.is_noisy.to_numpy()].mean():.3f}"
      f"   instance: {difficulty[inst.is_noisy.to_numpy()].mean():.3f}   (all reviews: {difficulty.mean():.3f})")

pd.set_option("display.max_colwidth", 300)
(inst[inst.is_noisy].assign(ambiguity=difficulty[inst.is_noisy.to_numpy()])
     .sort_values("ambiguity", ascending=False)[["id", "true_label", "label", "ambiguity", "text"]].head())

## 5. Frozen BERT embeddings (for the detector)
Frozen = BERT never sees our labels, so it can't memorise the noise. Computed once (~5-10 min on a T4) and cached.

In [ ]:
from pathlib import Path
from src.model import embed_texts

det_cfg = cfg["detection"]
cache = Path(cfg["paths"]["cache_dir"]) / f"emb_{det_cfg['embed_model']}_{len(train_df)}_{det_cfg['embed_max_length']}.npy"
features = embed_texts(train_df["text"].tolist(), det_cfg["embed_model"], det_cfg["embed_max_length"],det_cfg["embed_batch_size"], str(cache))
features.shape

## 6. Label-error detection (Level 1: data quality)
Cheap: seconds per run. Out-of-fold LogisticRegression probabilities → Cleanlab → compare flags with the
labels we actually corrupted. Expect lower recall on instance noise: ambiguous reviews are hard to judge.

In [ ]:
from src.pipeline import detect, plan_jobs

det_rows = []
for ntype, rate, seed in plan_jobs(cfg["noise"]):
    _, _, det = detect(train_df, features, rate, seed, det_cfg, ntype, difficulty)
    det_rows.append({"noise_type": ntype, "noise_rate": rate, "seed": seed, **det})

det_table = pd.DataFrame(det_rows)
det_table.groupby(["noise_type", "noise_rate"])[["n_true_errors", "n_flagged", "det_precision", "det_recall",
                                                 "det_f1", "review_reduction"]].mean().round(3)

### errors in the *original* IMDb labels
At 0% injected noise, anything Cleanlab flags is a candidate error in the real dataset. Read a few:
some will be genuinely mislabeled or ambiguous reviews.

In [ ]:
from src.cleaning import find_issues, get_oof_probs

labels0 = train_df["label"].to_numpy()
issues0 = find_issues(labels0, get_oof_probs(features, labels0, det_cfg["cv_folds"], cfg["seed"]))
suspects = pd.concat([train_df, issues0], axis=1).query("is_label_issue").sort_values("label_quality")
print(f"{len(suspects):,} suspicious labels in the original training subset")
pd.set_option("display.max_colwidth", 300)
suspects[["id", "label", "suggested_label", "label_quality", "text"]].head(10)

## 7. Train the SAME BERT on clean / noisy / cleaned data (Level 2: model quality)
Every run uses identical hyperparameters from `config.yaml`. Each finished run is appended to the results
CSV, so if Colab disconnects, just re-run the notebook: finished runs are skipped.

Cost on a T4 with 10k train reviews: ~6.5 min per run. Full config = 2 noise types × 3 rates × 3 seeds ×
3 datasets + 0% noise × 3 seeds × 3 datasets = **63 runs (~7 h)**, spread over several sessions.

In [ ]:
from src.model import fine_tune_and_predict
from src.pipeline import run_experiments

results = run_experiments(cfg, train_df, test_df, features, fine_tune_and_predict,results_file=cfg["paths"]["results_file"],issues_dir=cfg["paths"]["issues_dir"])
results.tail()

## 8. Results

In [ ]:
from src.evaluate import recovery_table, summarize, with_noise_type

results = with_noise_type(pd.read_csv(cfg["paths"]["results_file"]))
summarize(results, metrics=("accuracy", "precision", "recall", "f1"))

In [ ]:
recovery_table(results, "f1")

In [ ]:
from src.evaluate import plot_detection, plot_metric_vs_noise, plot_noise_type_comparison

fig_dir = cfg["paths"]["figures_dir"]
for ntype in sorted(set(results["noise_type"]) - {"none"}):
    plot_metric_vs_noise(results, "f1", f"{fig_dir}/f1_vs_noise_{ntype}.png", noise_type=ntype)
plot_noise_type_comparison(results, "f1", f"{fig_dir}/noise_type_comparison.png")
plot_detection(results, f"{fig_dir}/detection_vs_noise.png");

In [ ]:
from src.evaluate import plot_confusion

NTYPE, RATE, SEED = "instance", 0.2, cfg["noise"]["seeds"][0]
for name in ["noisy", "cleaned_remove", "cleaned_hybrid", "cleaned_relabel"]:
    match = results.query("noise_type == @NTYPE and noise_rate == @RATE and seed == @SEED and dataset == @name")
    if match.empty:
        print(f"skip {name}: not run yet")
        continue
    plot_confusion(match.iloc[0], f"{name} @ {RATE:.0%} {NTYPE} noise",
                   f"{fig_dir}/cm_{NTYPE}_{name}_{int(RATE*100)}.png")

## 9. Error analysis
Which flags were *wrong* (flagged, but we never corrupted them)? These are often reviews that are
genuinely ambiguous or mislabeled in the original IMDb, which is an interesting finding in itself.

In [ ]:
from pathlib import Path
from src.pipeline import issues_filename

NTYPE, RATE, seed = "instance", 0.2, cfg["noise"]["seeds"][0]
path = Path(cfg["paths"]["issues_dir"]) / issues_filename(NTYPE, RATE, seed)
if not path.exists():
    print(f"skip: {path.name} not created yet")
else:
    flagged = pd.read_csv(path)
    false_flags = flagged[~flagged["is_noisy"]]
    print(f"flagged: {len(flagged):,}   truly corrupted: {flagged.is_noisy.sum():,}   false flags: {len(false_flags):,}")
    display(false_flags[["id", "true_label", "label", "suggested_label", "label_quality", "text"]].head(10))

## 10. Conclusions
_Fill in after the runs finish. Template:_

- Random (symmetric) label noise at **X%** cost BERT only **A** F1 points; realistic (instance) noise cost **B**.
- Cleanlab detected **R%** of the corrupted labels with **P%** precision, flagging **N** of 10,000 reviews.
- Retraining the **same** BERT on the cleaned data recovered **Y%** of the F1 lost to realistic noise.

## 11. Download results
Bundles the results CSV, flagged-label files and figures into one zip. Unzip it into `results/` on your
laptop and commit from there (no GitHub token needed in Colab).

In [ ]:
import shutil
from google.colab import files

out = "results_bundle"
shutil.copytree(cfg["paths"]["figures_dir"], f"{out}/figures", dirs_exist_ok=True)
shutil.copy(cfg["paths"]["results_file"], f"{out}/experiments.csv")
shutil.copytree(cfg["paths"]["issues_dir"], f"{out}/label_issues", dirs_exist_ok=True)
shutil.make_archive(out, "zip", out)
files.download(f"{out}.zip")